# 🍕 Ilang pizza? on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho0912/ilang-pizza/blob/main/notebooks/ilang_pizza_colab.ipynb)

Predicts how many of your free-meetup RSVPs will actually show up, then how many pizzas to order.
This notebook runs the whole tool on a Colab GPU with **synthetic demo data**: made-up events and
people, shaped like Python Philippines meetups.

**Before you start:** *Runtime → Change runtime type → GPU*. A T4 is enough; pick L4 or A100 if your
plan includes compute units. Then run the cells from top to bottom.

> **Privacy:** don't upload raw Meetup, Luma, ti.to or Google Forms exports here. They contain
> names and emails. To try real data, upload the hashed copy described at the end.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU: Runtime → Change runtime type → GPU"

## 1. Install

In [ ]:
%pip install -q "ilang-pizza @ git+https://github.com/Jericho0912/ilang-pizza"
!ilang-pizza --help

## 2. Make the demo data

About 60 past events with check-ins, plus a couple of upcoming ones. Everything built from it is
labeled as demo data.

In [ ]:
!ilang-pizza demo --out data/demo
!ilang-pizza doctor --data data/demo

## 3. Backtest

Replays each past event using only the events before it, and compares TabPFN with "half will
show", the historical average and logistic regression. The first run downloads TabPFN v2's open
weights from Hugging Face.

In [ ]:
!ilang-pizza backtest --data data/demo --last-n 20

## 4. Forecast and plan the next event

In [ ]:
!ilang-pizza forecast --data data/demo
!ilang-pizza plan --data data/demo --capacity 80 --price 499

## 5. Optional: Gemma 4 for the emails and Ask Juan

Installs [Ollama](https://ollama.com) inside this runtime and pulls Gemma 4 E4B for NVIDIA GPUs
(`gemma4:e4b-it-q4_K_M`, about 6.6 GB; on a Mac, `gemma4:e4b` is an Apple MLX build). Gemma runs
in this VM, with no tunnel and nothing exposed to the internet. Skip this section and the drafts
use built-in templates instead.

In [ ]:
import os

LLM_TAG = "gemma4:e4b-it-q4_K_M"
os.environ["ILANG_PIZZA_LLM"] = LLM_TAG

!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve > ollama.log 2>&1 &
!sleep 5 && ollama pull $LLM_TAG

In [ ]:
!ilang-pizza draft --data data/demo --kind reconfirm --tone taglish
!ilang-pizza ask --data data/demo "Ilang pizza para sa next PyMeets?"

## 6. Optional: the web app, privately

Starts the Gradio app inside the runtime and opens it through Colab's own proxy, so only you,
signed in to this notebook, can reach it. Avoid `--share` with real data: those links are public.

In [ ]:
from google.colab import output

!nohup ilang-pizza ui --data data/demo --port 7860 > ui.log 2>&1 &
!sleep 30  # the first start loads TabPFN and Gradio; see ui.log if the window stays blank
output.serve_kernel_port_as_window(7860)

## Real PythonPH data (optional)

Keep the raw exports on your laptop. There, make a copy where names and emails are already
replaced by salted hashes:

```python
from ilang_pizza.data.loaders import load_dataset
from ilang_pizza.data.synth import write_dataset

write_dataset(load_dataset("data/pythonph"), "data/pythonph-hashed")
```

Zip it from a terminal (`cd data && zip -r pythonph-hashed.zip pythonph-hashed`), upload the zip
with the next cell, and keep `.ilang-pizza-salt` on your laptop. The
[data guide](https://github.com/Jericho0912/ilang-pizza/blob/main/docs/data-guide.md) covers the exports.

In [ ]:
import zipfile

from google.colab import files

uploaded = files.upload()  # choose pythonph-hashed.zip
for name in uploaded:
    zipfile.ZipFile(name).extractall("data")
!ilang-pizza backtest --data data/pythonph-hashed --last-n 20

## When you're done

*Runtime → Disconnect and delete runtime*, so the GPU stops using compute units. The demo data
and anything you uploaded are deleted with it.

*Built with PriorLabs-TabPFN.* Gemma 4 via [Ollama](https://ollama.com). Weather data by
[Open-Meteo.com](https://open-meteo.com/) for real events; the demo ships its own rain.